# LLM-JEPA Symbolic Regression: Usage Guide

This notebook provides an interactive guide to using the LLM-JEPA Symbolic Regression project. It covers data preparation, training, evaluation, and inference.

## 1. Setup (for Google Colab)

If you are running this on Google Colab, execute the following cell to clone the repository and install dependencies.

In [ ]:
import os
!git clone https://github.com/udohchuks/GSOC-LM-JEPA_for_Symbolic_Regression.git
%cd GSOC-LM-JEPA_for_Symbolic_Regression

!pip install -r requirements.txt

## 2. Dataset Preparation

Download and extract the AI Feynman dataset.

In [ ]:
import os
import tarfile
import urllib.request

data_dir = './data/'
tar_path = './Feynman_with_units.tar.gz'

if not os.path.exists(os.path.join(data_dir, 'Feynman_with_units')):
    print("Extracting dataset...")
    if not os.path.exists(tar_path):
        # Use dl=1 for direct download
        url = 'https://www.dropbox.com/s/7kgfr00qpokgz8w/Feynman_with_units.tar.gz?dl=1'
        urllib.request.urlretrieve(url, tar_path)
    
    with tarfile.open(tar_path) as tar:
        tar.extractall(data_dir)
    print("Extraction complete.")
else:
    print("Dataset already exists.")

### Verify Data Loading

Run the smoke test to ensure the CSV and data files are correctly mapped.

In [ ]:
%run run_test_load.py

## 3. Viewing Training Progress (TensorBoard)

### How it works in Colab
The `%tensorboard` magic command below launches a **background process** on the Colab server. 

**Crucial Tip:** Execute this cell **BEFORE** starting the training cell. Once the dashboard appears, you can proceed to the next cell. Even while the training cell is running (blocking execution), the TensorBoard window above will remain interactive and will automatically refresh every 30 seconds as the model writes new logs to disk.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

### Alternative: Access via External Tab (ngrok)
If you prefer to have TensorBoard open in a separate browser tab while you work, you can use `ngrok`. This is useful if the inline window feels too cramped.

*(Requires a free ngrok auth token from [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken))*:

In [ ]:
# !pip install pyngrok
# from pyngrok import ngrok
# # Replace 'YOUR_AUTH_TOKEN' with your actual token
# !ngrok authtoken YOUR_AUTH_TOKEN
# public_url = ngrok.connect(6006)
# print(f"TensorBoard available at: {public_url}")

## 4. Training

Training is managed by PyTorch Lightning. All hyperparameters are in `configs/base_config.yaml`.

In [ ]:
# Start training
!python -m training.train --config configs/base_config.yaml

## 5. Evaluation

Run the comprehensive evaluation suite on a trained checkpoint. This will compute precision, complexity, and robustness metrics.

In [ ]:
# Replace with your actual checkpoint path
checkpoint_path = './checkpoints/last.ckpt'
if os.path.exists(checkpoint_path):
    !python run_eval.py --ckpt {checkpoint_path} --output ./results/eval_results.json
else:
    print(f"Checkpoint not found at {checkpoint_path}. Please train the model first.")

### Viewing Evaluation Results

Detailed evaluation results are saved to `results/eval_results.json`. You can load and inspect them here:

In [ ]:
import json
results_path = './results/eval_results.json'
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        metrics = json.load(f)
    print(f"Exact Recovery Rate: {metrics.get('exact_recovery_rate')*100:.1f}%")
    print(f"Post-BFGS Mean R2: {metrics.get('mean_r2_post_bfgs'):.4f}")
    print(f"Mean Node Count: {metrics.get('mean_node_count'):.1f}")

## 6. Inference / Prediction

Use a trained model to generate symbolic formulas for specific AI Feynman equations or custom data.

In [ ]:
if os.path.exists(checkpoint_path):
    # Predict on equation I.6.2a
    !python predict.py --ckpt {checkpoint_path} --eq_id I.6.2a
else:
    print("Checkpoint not found.")

## 7. Output Files Guide

Here is a summary of the files and directories created during usage:

| Path | Description |
|---|---|
| `checkpoints/` | Contains the top-K model checkpoints (.ckpt) based on validation loss. |
| `tb_logs/` | TensorBoard logs for monitoring training history and SIGReg loss components. |
| `cache/` | Preprocessed data files to speed up subsequent loading. |
| `results/eval_results.json` | Detailed metrics from the evaluation suite (noise tolerance, data efficiency, etc.). |
| `lightning_logs/` | Default PyTorch Lightning log directory if not overridden. |